# 使用沙箱智能体迁移遗留代码库

代码现代化永远不会真正结束。过时的依赖、安全风险、合规压力和遗留模式会在大型代码库中不断累积，而一个庞大的迁移 PR 既难以审查，合并起来风险也很高。代码迁移智能体应当在受控环境中工作，一次只处理一个限定范围的任务：检查相关仓库、编辑文件、运行检查，然后返回补丁。

本 cookbook 使用 Agents SDK，并将运行框架（harness）置于沙箱*之外*：编排逻辑保留在受信任的宿主进程中，而 shell 命令和文件编辑则在隔离的执行环境中运行。这种分离让宿主侧的运行框架可以使用密钥、工具和外部服务，同时只向沙箱提供完成任务所需的文件和命令。

完成本 cookbook 后，你将能够：

- 将智能体运行框架置于运行 shell 命令和文件编辑的执行环境之外
- 将一项现代化工作拆分为任务大小的仓库分片
- 通过测试、检查、产物和审计日志验证每个分片
- 无需重写智能体即可更换沙箱提供商

本示例是一个包含两个服务的代码迁移。每个服务在各自的沙箱中运行，并返回各自的补丁包，这种形式同样可以用来分别创建 pull request 以供审查和 CI。在每个沙箱中，智能体会将一个 OpenAI 客户端封装从 [Chat Completions](https://platform.openai.com/docs/api-reference/chat) 迁移到 [Responses API](https://platform.openai.com/docs/api-reference/responses)。在此过程中，它会运行测试、修补应用和测试、运行编译检查、重新运行测试，并返回一份带补丁的类型化迁移报告。

我们将在本地使用 Docker 运行沙箱。特定于提供商的代码被隔离在沙箱创建环节，因此同样的运行框架和智能体可以指向 E2B 或 Cloudflare 等托管沙箱提供商，而无需修改 `SandboxAgent`、工具、manifest 或提示词。


## 架构：将沙箱作为工具

<img src="assets/sandbox-as-a-tool-architecture.svg" alt="Agents SDK harness running outside a swappable sandbox" width="900"/>

受信任的宿主进程拥有 Agents SDK 运行框架、工具、MCP 服务器、凭据、策略和审计。执行环境只接收当前任务限定范围的工作区，以及运行命令和编辑文件所需的面向沙箱的能力。

另一种模式是启动一个编码智能体，其运行框架、智能体循环、工具和文件系统全部位于沙箱内部。这种方式可行，但它会把编排和工具集成推入运行生成代码的同一环境中。

在本模式中，沙箱是运行框架在智能体需要文件系统、终端命令、测试运行或补丁时调用的对象。更广泛的智能体技术栈仍然保留在运行框架一侧。

流程：

1. 你的应用接收迁移请求，并将其拆分为任务大小的仓库分片。
2. 对于每个任务，宿主侧的 Agents SDK 运行框架启动一个智能体并创建一个全新的沙箱。
3. 运行框架将该任务的仓库和迁移说明暂存到沙箱中。
4. 智能体使用沙箱工具检查文件、编辑代码并运行测试。
5. 宿主接收该任务的报告和补丁，写入审计日志，删除沙箱，然后进入下一个任务。


## 环境要求

- Python 3.10+
- Docker，在本地运行以支持 Docker 沙箱示例
- 一个 OpenAI API 密钥，导出为 `OPENAI_API_KEY`
- 支持沙箱的 OpenAI Agents SDK
- 可选：托管沙箱提供商的凭据，例如 E2B 的 `E2B_API_KEY`，或 Cloudflare 的 Worker URL 和 API 密钥

请将 API 密钥保留在宿主环境中，不要将它们添加到挂载的仓库或沙箱 manifest 中。

## 安装依赖

克隆 cookbook 并进入本示例目录：

```bash
git clone https://github.com/openai/openai-cookbook.git
cd openai-cookbook/examples/agents_sdk/sandboxed-code-migration
```

从该目录打开 `sandboxed_code_migration_agent.ipynb`，并安装下面的依赖。在运行完整的智能体演示之前，请先启动 Docker。

In [ ]:
%pip install -r requirements.txt --quiet

## 导入宿主侧运行框架

导入本示例使用的小型宿主侧运行器。它负责创建沙箱会话、启动智能体循环、写入返回的产物、记录审计日志，并确保提供商凭据不会进入挂载的仓库。完整文件位于 [`src/run_migration_agent.py`](src/run_migration_agent.py)；notebook 会在下面的演练中摘取其中的重要部分。

In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

from agents import ModelSettings
from agents.mcp import MCPServer
from agents.sandbox import Manifest, SandboxAgent
from agents.sandbox.capabilities import ApplyPatch, Shell
from agents.sandbox.entries import LocalDir, LocalFile

EXAMPLE_ROOT = Path.cwd()
sys.path.insert(0, str(EXAMPLE_ROOT))

from src.run_migration_agent import (
    AGENT_INSTRUCTIONS,
    DEFAULT_MIGRATION_TASKS,
    DEVELOPER_INSTRUCTIONS,
    MigrationResult,
    MigrationTask,
    OPENAI_RESPONSES_MIGRATION_DOC_URL,
    run_migration_campaign,
)

migration_tasks = list(DEFAULT_MIGRATION_TASKS)
task = migration_tasks[0]

print("Migration campaign:")
for migration_task in migration_tasks:
    print(f"- {migration_task.name}: {migration_task.repo_path}")
print(f"\nInspecting first task: {task.name}")


## 1. 定义迁移任务

本 cookbook 在 `repo_fixtures/` 中包含两个小型示例仓库。如果你直接运行本 notebook，宿主运行框架会将每个示例仓库挂载到一个全新沙箱中的 `/workspace/repo`，并要求智能体遵循该仓库的 `MIGRATION.md`。

任务列表将每个迁移分片指向一个本地仓库：

```python
@dataclass(frozen=True)
class MigrationTask:
    name: str
    repo_path: Path

    @property
    def migration_brief_path(self) -> Path:
        return self.repo_path / "MIGRATION.md"

DEFAULT_MIGRATION_TASKS = (
    MigrationTask(
        name="support_reply_service",
        repo_path=EXAMPLE_ROOT / "repo_fixtures" / "support_reply_service",
    ),
    MigrationTask(
        name="case_summary_service",
        repo_path=EXAMPLE_ROOT / "repo_fixtures" / "case_summary_service",
    ),
)
```

要将其适配到你自己的代码库，请替换任务的 `repo_path` 并编辑其 `MIGRATION.md`。通用的运行提示词可以保持不变，因为它会告诉智能体遵循所挂载仓库中的说明。

在智能体接触任务之前，先检查一个任务：迁移说明、OpenAI 客户端封装以及应用调用点。

In [ ]:
print(task.migration_brief_path.read_text())

现在检查两个主要的代码目标：OpenAI 客户端封装，以及使用它的应用调用点。


In [ ]:
print((task.repo_path / "customer_support_bot" / "client.py").read_text())

In [ ]:
print((task.repo_path / "customer_support_bot" / "replies.py").read_text())

## 2. 在智能体编辑之前验证基线

在智能体编辑仓库之前，先自己运行一遍相同的测试。智能体在修改任何文件之前，也会在该任务的沙箱中运行这条基线测试命令。

In [ ]:
baseline = subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", "tests", "-t", "."],
    cwd=task.repo_path,
    text=True,
    capture_output=True,
    check=False,
)
print(baseline.stdout)
print(baseline.stderr)
assert baseline.returncode == 0

In [ ]:
check = subprocess.run(
    [sys.executable, "-m", "compileall", "-q", "customer_support_bot", "tests"],
    cwd=task.repo_path,
    text=True,
    capture_output=True,
    check=False,
)
print(check.stdout)
print(check.stderr)
assert check.returncode == 0

## 3. 暂存沙箱工作区

manifest 就是沙箱的边界。它告诉沙箱客户端要暂存哪些宿主文件，以及这些文件应出现在执行环境中的什么位置。这里，我们将智能体指令和一个任务仓库复制到 `/workspace` 中。

对于真实的迁移，只暂存目标检出代码、任务指令以及本次运行所需的文件。将凭据、客户存储和记忆保留在宿主运行框架中。正因如此，下面的辅助函数保持得很精简：它只暂存共享的智能体指令和当前迁移任务的仓库。


In [ ]:
def build_manifest(task: MigrationTask | None = None) -> Manifest:
    task = task or DEFAULT_MIGRATION_TASKS[0]
    return Manifest(
        root="/workspace",
        entries={
            "migration_agent/AGENTS.md": LocalFile(
                src=EXAMPLE_ROOT / "migration_agent" / "AGENTS.md"
            ),
            "repo": LocalDir(src=task.repo_path),
        },
    )

manifest = build_manifest(task)
print(f"manifest root: {manifest.root}")
for workspace_path in manifest.entries:
    print(workspace_path)


## 4. 定义沙箱智能体

智能体获得两项面向沙箱的能力：用于终端操作的 `Shell()` 和用于文件编辑的 `ApplyPatch()`。定义中的其他所有内容都保留在宿主运行框架中：指令、模型设置、MCP 服务器以及类型化的输出契约。


In [ ]:
def build_agent(
    *,
    model: str,
    manifest: Manifest,
    mcp_servers: list[MCPServer] | None = None,
) -> SandboxAgent:
    return SandboxAgent(
        name="Code Migration Agent",
        model=model,
        instructions=AGENT_INSTRUCTIONS,
        developer_instructions=DEVELOPER_INSTRUCTIONS,
        default_manifest=manifest,
        capabilities=[Shell(), ApplyPatch()],
        mcp_servers=list(mcp_servers or []),
        model_settings=ModelSettings(tool_choice="required"),
        output_type=MigrationResult,
    )

agent = build_agent(model="gpt-5.4", manifest=manifest)
print(agent.name)
print([capability.type for capability in agent.capabilities])
print(agent.output_type)


### 可选：连接宿主侧 MCP 服务器

由于运行框架在沙箱之外运行，它可以从受信任的宿主进程连接 MCP 服务器。沙箱不需要 MCP 凭据，也不需要广泛的网络访问权限。

该运行器可以选择从宿主运行框架连接公开的 OpenAI 文档 MCP。智能体可以使用这些文档上下文，而 shell 命令和补丁仍然在沙箱中运行。

对于迁移任务，请保持这一过程的确定性。获取经过批准的迁移指南，而不是让智能体在每次运行时都去搜索文档。

In [ ]:
OPTIONAL_OPENAI_DOCS_MCP_URL = "https://developers.openai.com/mcp"

print("Optional host-side MCP:")
print(f"  server: {OPTIONAL_OPENAI_DOCS_MCP_URL}")
print(f"  pinned doc: {OPENAI_RESPONSES_MIGRATION_DOC_URL}")
print("To opt in for the full run, set this before the agent cell:")
print(f'  os.environ["OPENAI_DOCS_MCP_URL"] = "{OPTIONAL_OPENAI_DOCS_MCP_URL}"')

## 5. 运行迁移活动

完整的运行是一个在宿主侧遍历迁移任务的循环。对于每个任务，运行框架构建 manifest 和智能体，创建一个全新的沙箱会话，并将该会话传入 `Runner.run_streamed`。任务完成后，宿主将返回的补丁包写入 `outputs/<task_name>/`，并在开始下一个任务之前删除沙箱。

```python
manifest = build_manifest(task)
agent = build_agent(model=model, manifest=manifest, mcp_servers=mcp_servers)
client, session = await create_sandbox(backend, manifest, docker_image=docker_image)

try:
    async with session:
        result = Runner.run_streamed(
            agent,
            [{"role": "user", "content": f"Task name: {task.name}\n\n{prompt}"}],
            max_turns=30,
            run_config=RunConfig(
                sandbox=SandboxRunConfig(session=session),
                workflow_name=f"Sandboxed code migration: {task.name} ({backend})",
                tracing_disabled=not enable_hosted_tracing,
            ),
        )
        async for event in result.stream_events():
            if event.type == "run_item_stream_event" and event.name in {"tool_called", "tool_output"}:
                append_audit_event(audit_log_path, {"event": event.name})
finally:
    await client.delete(session)
```

上面的代码片段是 `run_migration_task` 的核心；下面可运行的单元格会调用完整的辅助函数。该运行设置了保护开关，使 notebook 可以在不调用模型的情况下执行。当你想端到端运行基于 Docker 的迁移时，请将 `RUN_FULL_AGENT_DEMO` 改为 `True`。

In [ ]:
RUN_FULL_AGENT_DEMO = False
enable_hosted_tracing = False

# Optional: uncomment to let the host harness fetch the pinned Responses migration guide.
# os.environ["OPENAI_DOCS_MCP_URL"] = "https://developers.openai.com/mcp"

if RUN_FULL_AGENT_DEMO:
    campaign = await run_migration_campaign(
        tasks=migration_tasks,
        backend=os.getenv("SANDBOX_BACKEND", "docker"),
        model=os.getenv("OPENAI_MODEL", "gpt-5.4"),
        prompt=(
            "Migrate the mounted repo from Chat Completions to the Responses API. "
            "Follow migration_agent/AGENTS.md and repo/MIGRATION.md. "
            "Run the required baseline tests, patch the app and tests, "
            "run the check command, run the final tests, produce a diff, "
            "and return the structured migration result."
        ),
        docker_image=os.getenv("SANDBOX_DOCKER_IMAGE", "python:3.14-slim"),
        output_root=EXAMPLE_ROOT / "outputs",
        enable_hosted_tracing=enable_hosted_tracing,
    )
    for summary in campaign.task_summaries:
        print(f"{summary.task_name}: {summary.patch_path}")
else:
    print("Skipped. Set RUN_FULL_AGENT_DEMO=True to run the Docker-backed campaign.")

## 6. 检查返回的产物

宿主运行器会将每个任务的类型化结果写入磁盘。沙箱在运行结束后可以销毁；每个任务的报告、补丁、JSON 结果和审计日志都保留在 `outputs/<task_name>/` 中，活动汇总则位于 `outputs/batch_summary.json`。

In [ ]:
artifact_names = [
    "migration_report.md",
    "migration.patch",
    "migration_result.json",
    "migration_audit.jsonl",
]

for migration_task in migration_tasks:
    print(f"\n=== {migration_task.name} ===")
    task_output_dir = EXAMPLE_ROOT / "outputs" / migration_task.name
    for artifact_name in artifact_names:
        path = task_output_dir / artifact_name
        if path.exists():
            print(f"\n--- {path.name} ---")
            print(path.read_text()[:3000])
        else:
            print(f"not generated yet: {path}")

### 可选：验证生成的产物

在向用户展示补丁或将其应用到真实仓库之前，宿主可以检查每个返回的补丁、类型化结果和审计日志。这项评估是确定性的：它读取活动输出，如果任何任务没有产出预期的契约，就判定失败。


In [ ]:
if (EXAMPLE_ROOT / "outputs" / "batch_summary.json").exists():
    subprocess.run([sys.executable, "evals.py"], cwd=EXAMPLE_ROOT, check=True)
else:
    print("Skipped. Run the full agent demo before running artifact evals.")


## 7. 可选：更换沙箱提供商

本节展示三种沙箱后端：用于本地运行的 Docker、用于托管沙箱的 E2B，以及用于基于 Worker 的托管沙箱的 Cloudflare。模式是一样的：更换沙箱客户端，而不是更换智能体。

Docker：

```python
client = DockerSandboxClient(docker.from_env())
session = await client.create(
    manifest=manifest,
    options=DockerSandboxClientOptions(image="python:3.14-slim"),
)
```

E2B：

```python
client = E2BSandboxClient()
session = await client.create(
    manifest=manifest,
    options=E2BSandboxClientOptions(
        sandbox_type=E2BSandboxType.E2B,
    ),
)
```

从 CLI 运行 E2B：

```bash
export E2B_API_KEY="..."
python src/run_migration_agent.py --backend e2b
```

Cloudflare：

```python
client = CloudflareSandboxClient()
session = await client.create(
    manifest=manifest,
    options=CloudflareSandboxClientOptions(
        worker_url=os.environ["CLOUDFLARE_SANDBOX_WORKER_URL"],
        api_key=os.environ.get("CLOUDFLARE_SANDBOX_API_KEY"),
    ),
)
```

从 CLI 运行 Cloudflare：

```bash
export CLOUDFLARE_SANDBOX_WORKER_URL="https://..."
export CLOUDFLARE_SANDBOX_API_KEY="..."
python src/run_migration_agent.py --backend cloudflare
```

从 CLI 运行 Docker：

```bash
python src/run_migration_agent.py --backend docker
```

## 生产环境注意事项

生产代码应将编排、执行、数据访问和返回的输出置于彼此独立的信任边界之后。将每个迁移任务视为独立的审查单元。

| 边界 | 生产模式 |
| --- | --- |
| 运行框架 | 将编排、工具、凭据、策略和审计保留在宿主上。 |
| 沙箱 | 只暂存任务工作区。在其中运行命令和编辑。任务完成后将其销毁。 |
| 数据访问 | 通过宿主路由客户存储和网络访问，而不是直接经由沙箱。 |
| 输出 | 在向用户展示或应用更改之前，在宿主中验证沙箱输出。 |


## 追踪与 ZDR

本示例通过 `RunConfig.tracing_disabled=True` 在每次运行中禁用托管追踪。如果在运行本 cookbook 的 CLI 时希望启用，请传入 `--enable-hosted-tracing`。当你希望在整个进程范围内禁用追踪时，Agents SDK 也支持全局环境变量 `OPENAI_AGENTS_DISABLE_TRACING=1`。

## 后续步骤

要适配这一模式，请将 `migration_tasks` 替换为你自己的仓库、软件包或服务。为每个任务提供一份检出代码和一个 `MIGRATION.md`。保持验证命令明确，并返回补丁供审查，而不是在宿主侧自动应用更改。添加确定性评估来检查迁移契约，而不仅仅是检查测试套件是否通过。

当工作涉及许多软件包时（例如从 Jest 迁移到 Vitest），宿主运行框架可以利用仓库元数据或管理者智能体来规划分片。每个分片都应产出与本 cookbook 相同的内容：来自隔离沙箱的、经过验证的补丁、报告和审计轨迹。